# Colombo Stock Exchange — Web Data Collection

**DA 2009 — Data Collection Methods II · Assignment 2 (Group Project)**
University of Colombo

**Group:** 25ada072 - Pasindu · 25ada073 - Sasini · 25ada141 - Salaama · 24ada076 - Nuhan

---

This notebook shows how the project collects its data, using the same code as
the app. Run it from top to bottom.

We collect three things from https://www.cse.lk :

- share prices for every listed company
- details for individual companies
- announcements, and the PDF documents attached to them

In [1]:
# Run from the project folder so the src files can be imported.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 12)

import config
print("Website:", config.BASE_URL)

Website: https://www.cse.lk


---
## 1. Checking we are allowed — robots.txt

`robots.txt` is a file websites publish to tell automated programs which pages
they may visit. We read it before collecting anything.

*Written by 24ada076 - Nuhan*

In [2]:
from src.ethics import Scraper

scraper = Scraper()
print(scraper.robots_text)

User-agent: *
Disallow:
Disallow: /cgi-bin/
Sitemap: https://www.cse.lk/sitemap.xml



Reading that:

- `User-agent: *` — the rules apply to us
- `Disallow:` on its own blocks nothing
- `Disallow: /cgi-bin/` blocks one section
- there is no `Crawl-delay`, so the site does not ask us to wait at all

We wait 1.5 seconds between requests anyway.

In [3]:
# Check a few addresses against the rules.
for address in [config.API_BASE + "/tradeSummary",
                config.PAGES["trade_summary"]]:
    allowed, reason = scraper.can_fetch(address)
    print(f"{'allowed' if allowed else 'blocked':8} {address}")

print()
print("Sections the site asks bots to avoid:", scraper.blocked_paths())
print("Our delay:", config.DELAY, "seconds")

allowed  https://www.cse.lk/api/tradeSummary
allowed  https://www.cse.lk/equity/trade-summary

Sections the site asks bots to avoid: ['/cgi-bin/']
Our delay: 1.5 seconds


---
## 2. Trying requests and BeautifulSoup

This is the first thing we tried.

*Written by 24ada076 - Nuhan*

In [4]:
import requests
from bs4 import BeautifulSoup

url = "https://www.cse.lk/"
response = requests.get(url, headers={"User-Agent": config.USER_AGENT},
                        timeout=30)

print("Status code:", response.status_code)
print("HTML size  :", f"{len(response.text):,} bytes")

soup = BeautifulSoup(response.text, "html.parser")

print("Words found:", len(soup.get_text(strip=True)), "characters")
print("Tables      :", len(soup.find_all("table")))

Status code: 200
HTML size  : 25,071 bytes
Words found: 26 characters
Tables      : 0


About 25,000 bytes of HTML, and almost no words in it.

The CSE website builds its pages with JavaScript. The server sends a nearly
empty page and the browser fills in the prices afterwards. `requests`
downloads the HTML file but does not run JavaScript, so there is nothing in
it for BeautifulSoup to find.

BeautifulSoup itself is fine. Here it is on the sitemap, which is an ordinary
file:

In [5]:
from src.static_scraper import scrape_sitemap

sitemap = scrape_sitemap(scraper)
print(f"{len(sitemap)} pages read from the sitemap without any trouble")
sitemap.head()

15 pages read from the sitemap without any trouble


,Page,Updated
0,https://www.cse.lk/,daily
1,https://www.cse.lk/about-us/,monthly
2,https://www.cse.lk/vision-mission/,monthly
3,https://www.cse.lk/market-summary/,daily
4,https://www.cse.lk/indices/,daily


---
## 3. Getting the data the way the website does

We opened the CSE site in a browser, pressed F12 and looked at the **Network**
tab. The page loads its data from addresses like
`https://www.cse.lk/api/tradeSummary`, which return JSON.

*Written by 25ada072 - Pasindu*

In [6]:
from src.api_client import CSEApi

api = CSEApi(scraper=scraper)

print("Market:", api.market_status())

index = api.aspi()
print(f"ASPI index: {index['value']:,.2f} ({index['change']:+.2f})")

Market: Market Closed


ASPI index: 21,623.66 (+3.22)


In [7]:
# One request gives us every listed company.
prices = api.trade_summary()

print("Shape:", prices.shape)
prices[["symbol", "name", "price", "change", "percentageChange",
        "turnover"]].head(10)

Shape: (291, 23)


,symbol,name,price,change,percentageChange,turnover
0,ABAN.N0000,ABANS ELECTRICALS PLC,1060.25,47.0,4.638539,674797.5
1,AFSL.N0000,ABANS FINANCE PLC,86.70,-2.4,-2.693603,19502.4
2,AEL.N0000,ACCESS ENGINEERING PLC,79.90,0.8,1.011378,62792664.2
3,ACL.N0000,ACL CABLES PLC,98.30,2.4,2.502607,35338711.3
4,APLA.N0000,ACL PLASTICS PLC,168.25,2.5,1.508296,11971124.5
5,ACME.N0000,ACME PRINTING & PACKAGING PLC,4.10,-0.2,-4.651163,4213914.4
6,AGAL.N0000,AGALAWATTE PLANTATIONS PLC,49.10,1.4,2.935010,491842.5
7,AGST.N0000,AGSTAR PLC,13.60,0.0,0.000000,7575797.1
8,AGST.X0000,AGSTAR PLC,11.60,-0.2,-1.694915,24410.0
9,AHUN.N0000,AITKEN SPENCE HOTEL HOLDINGS PLC,88.40,-0.9,-1.007839,2707552.1


In [8]:
# Some addresses need a value sent with them.
info = api.company_info("SAMP.N0000")

for field in ["name", "symbol", "lastTradedPrice", "previousClose",
              "marketCap", "p12HiPrice", "p12LowPrice"]:
    print(f"  {field:18} {info.get(field)}")

  name               SAMPATH BANK PLC
  symbol             SAMP.N0000
  lastTradedPrice    140.0
  previousClose      139.75
  marketCap          164178106400.0
  p12HiPrice         167.0
  p12LowPrice        135.0


---
## 4. Cleaning the data

Data taken from a website is not ready to use straight away.

*Written by 25ada073 - Sasini*

In [9]:
from src.cleaner import check, clean, compare, describe

before = describe(prices)
tidy, changes = clean(prices)
after = describe(tidy)

print("What we changed:")
for change in changes:
    print("  -", change)

What we changed:
  - Turned 1 column(s) of large numbers back into dates: lastTradedTime
  - Removed extra spaces from the text columns
  - Moved the most useful columns to the front


The biggest problem is the dates. CSE sends them as the number of
milliseconds since 1970, which pandas would otherwise treat as an ordinary
number.

In [10]:
from src.api_client import to_datetime

print("As CSE sends it:", 1786440420412)
print("What it means  :", to_datetime(1786440420412))

As CSE sends it: 1786440420412
What it means  : 2026-08-11 14:57:00.412000


In [11]:
compare(before, after)

,Check,Before,After
0,Rows,291,291
1,Columns,23,23
2,Empty cells,15,15
3,Empty columns,0,0
4,Duplicate rows,0,0
5,Number columns,19,18
6,Date columns,0,1


In [12]:
# Cleaning fixes the shape of the data. These checks ask whether the
# values make sense - a share price of -5 rupees would be tidy and still
# completely wrong.
check(tidy)

,Check,Result,Detail
0,Rows collected,OK,291 rows
1,No negative prices,OK,All prices are zero or above
2,Price changes look sensible,OK,Nothing moved more than 100%
3,Every row has a symbol,OK,All rows have one
4,No company appears twice,OK,Each company appears once
5,How complete is the data,OK,99.8% of cells have a value


---
## 5. Announcements and their documents

We do not know which documents exist, so we read the list of circulars, work
out the address of each PDF, and follow those addresses.

*Written by 25ada073 - Sasini*

In [13]:
from src.crawler import find_documents

documents = find_documents(api)
documents[["Title", "Date"]]

,Title,Date
0,DFCC BANK PLC (“BANK”) – ISSUANCE OF BASEL III...,03 Sep 2026 03:17:56 PM
1,EMPLOYEE SHARE OPTION SCHEMES,01 Sep 2026 05:05:33 PM
2,THE REVISED FORMAT FOR DISCLOSURE REGARDING FU...,21 Aug 2026 03:28:55 PM
3,DE-LISTING OF THE ORDINARY VOTING SHARES OF AS...,06 Aug 2026 09:21:30 AM
4,SAMPATH BANK PLC (“BANK”) – BASEL III COMPLIAN...,04 Aug 2026 08:44:27 PM


In [14]:
from src.crawler import download_documents

downloaded = download_documents(documents, limit=5, scraper=scraper)
downloaded

,Title,Result,File
0,DFCC BANK PLC (“BANK”) – ISSUANCE OF BASEL III...,Already saved,14436_DFCC BANK PLC BANK ISSUANCE OF BASEL II...
1,EMPLOYEE SHARE OPTION SCHEMES,Already saved,14434_EMPLOYEE SHARE OPTION SCHEMES.pdf
2,THE REVISED FORMAT FOR DISCLOSURE REGARDING FU...,Already saved,14396_THE REVISED FORMAT FOR DISCLOSURE REGARD...
3,DE-LISTING OF THE ORDINARY VOTING SHARES OF AS...,Already saved,14364_DE-LISTING OF THE ORDINARY VOTING SHARES...
4,SAMPATH BANK PLC (“BANK”) – BASEL III COMPLIAN...,Already saved,14361_SAMPATH BANK PLC BANK BASEL III COMPLIA...


---
## 6. Reading the PDFs

There are two kinds of PDF. A text based one was made by a computer and the
letters are real text. A scanned one is a photograph of a printed page, and
there is no text inside it to extract.

We use **pdfplumber** for the text and layout, and **PyPDF2** for the file
details.

*Written by 24ada076 - Nuhan*

In [15]:
from src.pdf_extractor import check_all_pdfs

check_all_pdfs()

,File,Pages,Characters,Type,Needs OCR
0,14336_EXTENSION OF THE FINAL SUSPENSION DATE F...,1,3267,Text based,No
1,14354_COMMERCIAL BANK OF CEYLON PLC BANK BASE...,1,1383,Text based,No
2,14358_EMPLOYEE SHARE OPTION SCHEMES.pdf,1,896,Text based,No
3,14361_SAMPATH BANK PLC BANK BASEL III COMPLIA...,1,955,Text based,No
4,14364_DE-LISTING OF THE ORDINARY VOTING SHARES...,1,0,Scanned image,Yes
5,14396_THE REVISED FORMAT FOR DISCLOSURE REGARD...,4,1530,Scanned image,Yes
6,14434_EMPLOYEE SHARE OPTION SCHEMES.pdf,1,996,Text based,No
7,14436_DFCC BANK PLC BANK ISSUANCE OF BASEL II...,2,1699,Text based,No


Look at the row with almost no characters. That file is about 190 KB and is
clearly full of writing when you open it — but there is no text inside it,
because it is a scan.

In [16]:
from src.pdf_extractor import list_pdfs, pages_table, read_pdf

# A document that worked normally.
for path in list_pdfs():
    result = read_pdf(path)
    if result["ok"] and not result["scanned_pages"]:
        break

print(result["summary"])
print()
for key, value in result["info"].items():
    print(f"  {key:14} {value}")
print()
print(result["text"][:500])

1 page(s) read, 3,267 characters extracted. Tables found: 0.

  Pages          1
  Title          not set
  Author         Nirmalee Ganegoda
  Created with   Microsoft® Word for Microsoft 365

----- Page 1 -----
Central Depository Systems (Pvt) Ltd.
Fully owned subsidiary of the Colombo Stock Exchange
Company No: PV 1270
# 04-01, West Block
World Trade Center
24 July 2026
Echelon Square
Colombo 00100, Sri Lanka
www.cds.lk
CIRCULAR NO:05-07-2026
To: All CEOs and Heads of Depository Participants
Dear Sir/Madam,
EXTENSION OF THE FINAL SUSPENSION DATE FOR CDS ACCOUNTS WITHOUT MANDATORY
KEY INFORMATION
The Central Depository Systems (Pvt) Ltd. (CDS) refers to its ongoing initiative to obta


---
## 7. OCR — reading the scanned document

OCR turns a picture of text into text the computer can use. There are four
steps: turn the page into an image, clean the image up, read the letters,
then tidy the result.

*Written by 25ada141 - Salaama*

This needs the Tesseract program installed, not just the Python package.

In [17]:
from src.ocr_extractor import tesseract_ready

ready, message = tesseract_ready()
print(message)

Tesseract is not installed on this computer.

To install it:
  1. Download it from https://github.com/UB-Mannheim/tesseract/wiki
  2. Install it to C:\Program Files\Tesseract-OCR\
  3. Tick the option to add it to the system PATH
  4. Restart this app

The Python part (pip install pytesseract) is already done.


In [18]:
from src.ocr_extractor import ocr_pdf_page

# Find the document that gave us no text.
scanned = None
for path in list_pdfs():
    result = read_pdf(path)
    if result["ok"] and result["scanned_pages"]:
        scanned = path
        break

if scanned is None:
    print("None of the PDFs we have are scans.")
elif not ready:
    print("Install Tesseract to run this step.")
else:
    print("Reading:", scanned.name)
    ocr = ocr_pdf_page(scanned, page_number=1)

    if ocr["ok"]:
        print(f"Characters read   : {ocr['characters']:,}")
        print(f"Average confidence: {ocr['average_confidence']}%")
        print()
        print(ocr["text"][:700])
    else:
        print(ocr["error"])

Install Tesseract to run this step.


OCR is not perfect. It runs words together and misreads logos as letters,
which is why we also record how confident it was about each word.

---
## 8. Charts and saving the data

*Written by 25ada141 - Salaama*

In [19]:
from src.analysis import market_figures, summary

for name, value in market_figures(tidy).items():
    print(f"  {name:12} {value}")

print()
print(summary(tidy, index))

  Companies    291
  Rose         107
  Fell         112
  No change    72
  Turnover     Rs 1.36 bn

The All Share Price Index rose to 21,623.66 (+3.22 points). Out of 291 companies traded, 107 rose, 112 fell and 72 finished unchanged, so more companies fell than rose. Total turnover was Rs 1.36 bn. This is one trading day, collected for a university assignment. It is not investment advice.


In [20]:
from src.analysis import chart_movers

chart_movers(tidy, dark=False)

In [21]:
from src.storage import export

files = export(tidy, "notebook_prices", formats=["csv", "json"])
for path in files:
    print("saved:", path.name)

print()
print("Reading the CSV back, skipping the comment lines at the top:")
pd.read_csv(files[0], comment="#").head(3)

saved: notebook_prices_20260907_232540.csv
saved: notebook_prices_20260907_232540.json

Reading the CSV back, skipping the comment lines at the top:


,symbol,name,price,change,percentageChange,quantity,...,marketCapPercentage,open,closingPrice,crossingVolume,crossingTradeVol,status
0,ABAN.N0000,ABANS ELECTRICALS PLC,1060.25,47.0,4.638539,77,...,0.0,1015.0,1060.25,637,20,0
1,AFSL.N0000,ABANS FINANCE PLC,86.70,-2.4,-2.693603,50,...,0.0,86.1,86.70,225,11,3
2,AEL.N0000,ACCESS ENGINEERING PLC,79.90,0.8,1.011378,50,...,0.0,79.1,79.90,784931,298,0


---
## 9. What it cost the website

Every request is written to `logs/request_log.csv`.

In [22]:
log = pd.read_csv(config.REQUEST_LOG)

print("Requests recorded:", len(log))
log.tail(8)[["time", "method", "url", "status", "delay_seconds"]]

Requests recorded: 111


,time,method,url,status,delay_seconds
103,2026-09-07 22:36:09,POST,https://www.cse.lk/api/allSectors,skipped,1.5
104,2026-09-07 22:36:15,GET,https://www.cse.lk/api/allSecurityCode,skipped,1.5
105,2026-09-07 22:36:37,POST,https://www.cse.lk/api/approvedAnnouncement,skipped,1.5
106,2026-09-07 23:25:29,POST,https://www.cse.lk/api/marketStatus,200,1.5
107,2026-09-07 23:25:30,POST,https://www.cse.lk/api/aspiData,200,1.5
108,2026-09-07 23:25:32,POST,https://www.cse.lk/api/tradeSummary,200,1.5
109,2026-09-07 23:25:33,POST,https://www.cse.lk/api/companyInfoSummery,200,1.5
110,2026-09-07 23:25:35,POST,https://www.cse.lk/api/circularAnnouncement,200,1.5


---
## Summary

| Step | Tool |
|---|---|
| Checking we are allowed | `robots.txt`, `RobotFileParser` |
| Getting the HTML | `requests`, BeautifulSoup |
| Getting the data | the addresses the site itself uses, read with `requests` and pandas |
| Running JavaScript when needed | Selenium |
| Finding and downloading documents | our crawler, and a Scrapy spider |
| Reading PDFs | PyPDF2, pdfplumber |
| Reading scanned pages | PIL, OpenCV, pytesseract |
| Cleaning and checking | pandas |
| Saving | CSV, Excel, JSON |

**Who wrote what**

| Member | Part |
|---|---|
| 24ada076 - Nuhan | robots.txt and requests, BeautifulSoup, reading PDFs |
| 25ada072 - Pasindu | getting the data from the site, Selenium |
| 25ada073 - Sasini | crawling for documents, the Scrapy spider, cleaning |
| 25ada141 - Salaama | OCR, saving the files, charts |

---

*Data source: Colombo Stock Exchange (https://www.cse.lk). Collected for
academic purposes only. Not investment advice.*